# NonLinearTime -- Kaggle backup runner

**Only use this if Groq/Featherless are both unavailable.** This notebook reruns the same graph pipeline / direct baseline / AWT-F1 scoring code from the project (`pipeline/`, `metrics/`) unchanged -- it does not reimplement the method, it only swaps in a locally-hosted open model as the `LLMClient` instead of an API.

## Setup (do this before running)
1. Create a new Kaggle Notebook, enable a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or P100).
2. Add Data -> Upload -> upload `nonlineartime_pipeline.zip` (packaged alongside this notebook) as a new Kaggle Dataset. Note the dataset's input path, usually `/kaggle/input/<dataset-slug>/`.
3. Set `DATASET_DIR` in the first code cell to that path.
4. Set `STORY_IDS` and `TASKS` below to whichever stories/conditions actually failed elsewhere -- running everything here is slower than the API path, so only backfill what you need.
5. Run all cells. Progress prints per story/stage as it goes, same style as the project's own `scripts/generate_*.py`.
6. Last cell zips `/kaggle/working/predictions/` -- download it and drop the files into `web/src/data/predictions/` (and `data/stories/` predictions dir, if the project mirrors it) on your machine, replacing the stale files with the same names.

In [ ]:
# ---- config ----
DATASET_DIR = "/kaggle/input/nonlineartime-pipeline"  # <-- change to your uploaded dataset's path
STORY_IDS = ["story_05"]  # which stories to (re)run -- edit this list
TASKS = ["counterfactual"]  # only counterfactual is still missing -- direct already succeeded
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"  # same model family as the paid Featherless run (Qwen2.5-72B), smaller so it fits a free Kaggle GPU
MAX_NEW_TOKENS = 4096  # lowered from 8192: the first attempt at this task took ~1h/retry and still produced malformed JSON ~23,600 characters in -- no constrained decoding means an 8192-token budget gives the model too much room to ramble past valid JSON structure, and this story's edge list doesn't need that many tokens anyway
MAX_RETRIES = 3  # lowered from 6 -- fail fast and report back rather than silently burning hours of GPU quota on a call that isn't working
OUT_DIR = "/kaggle/working/predictions"


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import os, sys, json, re

assert os.path.isdir(DATASET_DIR), (
    f"DATASET_DIR {DATASET_DIR!r} not found -- check Add Data on the right panel "
    "for the actual mounted path and update the config cell above."
)
sys.path.insert(0, DATASET_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

from pipeline.graph_pipeline import run_graph_pipeline
from pipeline.direct_baseline import run_direct_baseline
from metrics.allen_relations import edge_score
from metrics.awt_f1 import awt_f1, memorization_gap

print("pipeline/metrics imported from", DATASET_DIR)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"loading {MODEL_NAME} in 4-bit ...", flush=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
print("model loaded", flush=True)

## Local LLM client

Mirrors `pipeline/llm_client.py`'s `_OpenAICompatibleClient.complete()` contract exactly: same retry-on-invalid-JSON / retry-on-missing-required-key loop, just generating locally instead of over HTTP. No constrained/guided decoding is used here (keeps the dependency footprint small and matches the project's existing retry-based robustness pattern instead of introducing a new library) -- if the model's JSON gets rejected it is simply asked again, up to `MAX_RETRIES` times, with the validation error appended to the prompt.

In [ ]:
class LocalHFClient:
    def __init__(self, model, tokenizer, max_new_tokens=MAX_NEW_TOKENS, max_retries=MAX_RETRIES):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.max_retries = max_retries
        self._calls = []

    def _generate(self, system, user_prompt):
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_prompt},
        ]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    @staticmethod
    def _extract_json(raw):
        # Models often wrap JSON in ```json fences or add stray prose; pull out
        # the first top-level {...} block rather than assuming a clean reply.
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            raise json.JSONDecodeError("no JSON object found", raw, 0)
        return json.loads(match.group(0))

    def complete(self, prompt, schema):
        system = (
            "You are a precise information-extraction assistant. Respond with a single "
            "JSON object and nothing else, matching this JSON schema exactly. Every key "
            "listed in the schema's top-level `required` array MUST be present in your "
            "output even if its value is an empty array or list -- never omit a required "
            "key just because you have nothing to report for it.\n" + json.dumps(schema)
        )
        user_prompt = prompt
        last_error = None

        for attempt in range(self.max_retries + 1):
            print(f"    [LocalHFClient] attempt {attempt + 1}/{self.max_retries + 1}...", flush=True)
            raw = self._generate(system, user_prompt)
            self._calls.append({"prompt": prompt, "schema": schema, "raw_response": raw})
            try:
                parsed = self._extract_json(raw)
            except json.JSONDecodeError as exc:
                last_error = f"invalid JSON: {exc}"
                user_prompt = f"{prompt}\n\nYour previous reply was invalid ({last_error}). Return ONLY valid JSON matching the schema."
                continue

            missing = [key for key in schema.get("required", []) if key not in parsed]
            if missing:
                last_error = f"missing required keys: {missing}"
                user_prompt = f"{prompt}\n\nYour previous reply was invalid ({last_error}). Return ONLY valid JSON matching the schema."
                continue

            return parsed

        raise RuntimeError(f"LocalHFClient failed after {self.max_retries + 1} attempts: {last_error}")

client = LocalHFClient(model, tokenizer)
print("LocalHFClient ready")

## Run the requested stories/conditions

Reuses the exact scoring/output shape of `scripts/generate_real_predictions.py`, `generate_counterfactual_predictions.py`, and `generate_direct_baseline_predictions.py` so the resulting JSON files are drop-in replacements for `web/src/data/predictions/*.json`.

In [ ]:
STORIES_DIR = os.path.join(DATASET_DIR, "data", "stories")

with open(os.path.join(STORIES_DIR, "manifest.json"), encoding="utf-8") as f:
    manifest = {e["id"]: e for e in json.load(f)["stories"]}


def load_story(story_dir, variant):
    path = os.path.join(STORIES_DIR, story_dir, f"{variant}.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def passage_from_story(story):
    by_id = {e["id"]: e for e in story["events"]}
    ordered_ids = sorted(by_id, key=lambda i: int(i[1:]))
    return [by_id[i]["text"] for i in ordered_ids]


def scored_edges(gold_edges, predicted_edges):
    predicted_map = {(e["node_i"], e["node_j"]): e["allen_relation"] for e in predicted_edges}
    predicted_map.update({(e["node_j"], e["node_i"]): e["allen_relation"] for e in predicted_edges})
    scored = []
    for e in gold_edges:
        key = (e["node_i"], e["node_j"])
        predicted_relation = predicted_map.get(key)
        score = edge_score(e["allen_relation"], predicted_relation) if predicted_relation else 0.0
        scored.append({
            "node_i": e["node_i"],
            "node_j": e["node_j"],
            "gold_relation": e["allen_relation"],
            "predicted_relation": predicted_relation,
            "score": score,
        })
    return scored


def run_real(story_id):
    entry = manifest[story_id]
    story = load_story(entry["dir"], "original")
    passage = passage_from_story(story)
    print(f"-- {story_id} real ({len(story['events'])} events) --", flush=True)

    result = run_graph_pipeline(passage, client)
    result["model"] = MODEL_NAME

    scores = awt_f1(story, result)
    result["scored_edges"] = scored_edges(story["edges"], result["edges"])
    result["awt_f1_scores"] = scores

    unrepaired = dict(result, edges=result["raw_edges"], convergence_points=result["raw_convergence_points"])
    scores_unrepaired = awt_f1(story, unrepaired)
    result["scored_edges_unrepaired"] = scored_edges(story["edges"], result["raw_edges"])
    result["awt_f1_scores_unrepaired"] = scores_unrepaired

    out_path = os.path.join(OUT_DIR, f"{entry['dir']}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    print(f"{story_id}: awt_f1={scores['awt_f1']:.3f} "
          f"(relation={scores['relation_score']:.3f}, "
          f"thread={scores['thread_attribution_accuracy']:.3f}, "
          f"convergence={scores['convergence_f1']:.3f}) -> {out_path}", flush=True)


def run_counterfactual(story_id):
    entry = manifest[story_id]
    story = load_story(entry["dir"], "counterfactual")
    passage = passage_from_story(story)
    print(f"-- {story_id} counterfactual ({len(story['events'])} events) --", flush=True)

    result = run_graph_pipeline(passage, client)
    result["model"] = MODEL_NAME

    scores = awt_f1(story, result)
    result["scored_edges"] = scored_edges(story["edges"], result["edges"])
    result["awt_f1_scores"] = scores

    out_path = os.path.join(OUT_DIR, f"{entry['dir']}_counterfactual.json")

    original_path = os.path.join(OUT_DIR, f"{entry['dir']}.json")
    if not os.path.exists(original_path):
        # Also check the packaged web predictions folder in case only the
        # counterfactual run is being backfilled here.
        alt = os.path.join(DATASET_DIR, "predictions", f"{entry['dir']}.json")
        original_path = alt if os.path.exists(alt) else None
    if original_path:
        with open(original_path, encoding="utf-8") as f:
            original_result = json.load(f)
        if "awt_f1_scores" in original_result:
            gap = memorization_gap(original_result["awt_f1_scores"], scores)
            result["memorization_gap"] = gap

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    print(f"{story_id}: awt_f1={scores['awt_f1']:.3f} "
          f"(relation={scores['relation_score']:.3f}, "
          f"thread={scores['thread_attribution_accuracy']:.3f}, "
          f"convergence={scores['convergence_f1']:.3f}) -> {out_path}", flush=True)


def normalize_id(node_id):
    node_id = str(node_id)
    return node_id if node_id.startswith("s") else f"s{node_id}"


def derive_edges_from_order(global_order, gold_edges):
    position = {node_id: i for i, node_id in enumerate(global_order)}
    derived = []
    for e in gold_edges:
        i, j = e["node_i"], e["node_j"]
        if i not in position or j not in position:
            continue
        relation = "before" if position[i] < position[j] else "after"
        derived.append({"node_i": i, "node_j": j, "allen_relation": relation})
    return derived


def run_direct(story_id):
    entry = manifest[story_id]
    story = load_story(entry["dir"], "original")
    passage = passage_from_story(story)
    print(f"-- {story_id} direct ({len(story['events'])} events) --", flush=True)

    result = run_direct_baseline(passage, client)
    for event in result["events"]:
        event["id"] = normalize_id(event["id"])
    result["global_order"] = [normalize_id(i) for i in result["global_order"]]
    result["convergence_points"] = [
        [normalize_id(i) for i in pair] for pair in result["convergence_points"]
    ]
    result["model"] = MODEL_NAME

    derived_edges = derive_edges_from_order(result["global_order"], story["edges"])
    scoring_view = {
        "events": result["events"],
        "edges": derived_edges,
        "convergence_points": result["convergence_points"],
    }
    scores = awt_f1(story, scoring_view)
    result["derived_edges"] = derived_edges
    result["scored_edges"] = scored_edges(story["edges"], derived_edges)
    result["awt_f1_scores"] = scores

    out_path = os.path.join(OUT_DIR, f"{entry['dir']}_direct.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    print(f"{story_id}: awt_f1={scores['awt_f1']:.3f} "
          f"(relation={scores['relation_score']:.3f}, "
          f"thread={scores['thread_attribution_accuracy']:.3f}, "
          f"convergence={scores['convergence_f1']:.3f}) -> {out_path}", flush=True)


RUNNERS = {"real": run_real, "counterfactual": run_counterfactual, "direct": run_direct}

for story_id in STORY_IDS:
    for task in TASKS:
        try:
            RUNNERS[task](story_id)
        except Exception as exc:
            print(f"!! {story_id}/{task} FAILED: {exc}", flush=True)

print("\nAll requested runs attempted -- see per-story lines above for results/failures.")

## Package results for download

In [ ]:
import shutil

zip_path = shutil.make_archive("/kaggle/working/kaggle_predictions", "zip", OUT_DIR)
print(f"Zipped results -> {zip_path}")
print("Download it from the notebook's Output panel, then unzip into your local")
print("web/src/data/predictions/ (overwriting the matching stale files).")